In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ajinkyasupekar/fruit-vegetable-shelf-life-7500-xgboost/fruit_vegetable_shelf_life_7500_xgboost.csv


In [2]:
import sys
import subprocess

# ------------------------------------------------------------------
# STEP 1: AUTOMATIC PACKAGE INSTALLATION
# ------------------------------------------------------------------
required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'xgboost': 'xgboost',
    'sklearn': 'scikit-learn',
    'openpyxl': 'openpyxl',
    'joblib': 'joblib'
}

print("Checking required packages...")
for module_name, pip_name in required_packages.items():
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing package: '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

print("All required packages are installed and ready!\n")

# ------------------------------------------------------------------
# STEP 2: IMPORTS
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# ------------------------------------------------------------------
# STEP 3: LOAD & PREPROCESS DATASET
# ------------------------------------------------------------------
file_path = "/kaggle/input/datasets/ajinkyasupekar/fruit-vegetable-shelf-life-7500-xgboost/fruit_vegetable_shelf_life_7500_xgboost.csv"
print(f"Loading dataset from: {file_path}...")

df = pd.read_csv(file_path)
print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns")

target_col = 'Remaining_Shelf_Life_Days'
ignore_cols = [target_col, 'Status_Label'] if 'Status_Label' in df.columns else [target_col]

X = df.drop(columns=[col for col in ignore_cols if col in df.columns])
y = df[target_col]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical features: {categorical_cols}")
print(f"Numerical features: {numerical_cols}")

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

if categorical_cols:
    X[categorical_cols] = encoder.fit_transform(X[categorical_cols])

# ------------------------------------------------------------------
# STEP 4: TRAIN / TEST SPLIT
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------------------------------------
# STEP 5: TRAIN XGBOOST REGRESSOR
# ------------------------------------------------------------------
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("\nStarting XGBoost training...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# ------------------------------------------------------------------
# STEP 6: EVALUATE & SAVE ARTIFACTS
# ------------------------------------------------------------------
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\n========================================")
print("       MODEL EVALUATION RESULTS       ")
print("========================================")
print(f"Mean Absolute Error (MAE) : {mae:.3f} days")
print(f"Root Mean Squared Error   : {rmse:.3f} days")
print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%")
print("========================================\n")

model.save_model('xgb_shelf_life_model.json')
joblib.dump(encoder, 'xgb_categorical_encoder.pkl')
joblib.dump(X.columns.tolist(), 'xgb_feature_names.pkl')

print("Success! Saved artifacts:")
print(" - 'xgb_shelf_life_model.json'")
print(" - 'xgb_categorical_encoder.pkl'")
print(" - 'xgb_feature_names.pkl'")

print("-------------------------------------------------------")


Checking required packages...
All required packages are installed and ready!

Loading dataset from: /kaggle/input/datasets/ajinkyasupekar/fruit-vegetable-shelf-life-7500-xgboost/fruit_vegetable_shelf_life_7500_xgboost.csv...
Dataset successfully loaded: 7500 rows, 7 columns
Categorical features: ['Fruit_Vegetable', 'Storage_Condition']
Numerical features: ['Temperature_C', 'Humidity_Pct', 'Spoilage_Pct']

Starting XGBoost training...
[0]	validation_0-rmse:19.38308
[50]	validation_0-rmse:5.09425
[100]	validation_0-rmse:3.18796
[150]	validation_0-rmse:2.84251
[200]	validation_0-rmse:2.75969
[250]	validation_0-rmse:2.72104
[299]	validation_0-rmse:2.70861

       MODEL EVALUATION RESULTS       
Mean Absolute Error (MAE) : 1.822 days
Root Mean Squared Error   : 2.709 days
R² Score (Accuracy)       : 98.21%

Success! Saved artifacts:
 - 'xgb_shelf_life_model.json'
 - 'xgb_categorical_encoder.pkl'
 - 'xgb_feature_names.pkl'
-------------------------------------------------------
